https://colab.research.google.com/drive/1How_tOr6giSl6qAVWdGVjXL1lac7EvzU

1. Install and load package

In [ ]:
# Install sqldf package for running SQL queries inside R
install.packages("sqldf")

# Load sqldf library
library(sqldf)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘gsubfn’, ‘proto’, ‘RSQLite’, ‘chron’


Loading required package: gsubfn

Loading required package: proto

Warning message:
“no DISPLAY variable so Tk is not available”
Loading required package: RSQLite



This code installs and loads sqldf, which allows SQL queries to be written directly on R data frames.

2. Load datasets from GitHub

In [ ]:
# Base GitHub raw link where all CSV files are stored
base_url <- "https://raw.githubusercontent.com/maazmohammed626-ctrl/northstar-database-analytics/refs/heads/main/northstar_dataset/"

# Load all NorthStar CSV files into R data frames
orders <- read.csv(paste0(base_url, "orders.csv"))
deliveries <- read.csv(paste0(base_url, "deliveries.csv"))
complaints <- read.csv(paste0(base_url, "complaints.csv"))
customers <- read.csv(paste0(base_url, "customers.csv"))
drivers <- read.csv(paste0(base_url, "drivers.csv"))
vehicles <- read.csv(paste0(base_url, "vehicles.csv"))
incidents <- read.csv(paste0(base_url, "incidents.csv"))
app_events <- read.csv(paste0(base_url, "app_events.csv"))
hubs <- read.csv(paste0(base_url, "hubs.csv"))

This code loads all required datasets directly from GitHub, making the notebook reproducible and linked to the submitted repository.

3. Check table structure

In [ ]:
# Display column names for each important dataset
colnames(orders)
colnames(deliveries)
colnames(complaints)
colnames(customers)
colnames(drivers)
colnames(vehicles)
colnames(incidents)
colnames(app_events)
colnames(hubs)

[1] "order_id"              "customer_id"           "service_type"         
 [4] "order_created_at"      "promised_window_hours" "pickup_zone"          
 [7] "dropoff_zone"          "priority_level"        "order_value"          
[10] "booking_channel"       "special_handling_flag"

[1] "delivery_id"                   "order_id"                     
 [3] "driver_id"                     "vehicle_id"                   
 [5] "hub_id"                        "dispatch_time"                
 [7] "delivery_completed_at"         "delivery_status"              
 [9] "route_distance_km"             "manual_route_override_count"  
[11] "proof_of_completion_missing"   "customer_rating_post_delivery"
[13] "fuel_or_charge_cost"

[1] "complaint_id"        "customer_id"         "order_id"           
 [4] "complaint_type"      "channel"             "severity"           
 [7] "created_at"          "status"              "resolution_days"    
[10] "compensation_amount"

[1] "customer_id"          "age"                  "home_zone"           
[4] "customer_type"        "signup_date"          "loyalty_score"       
[7] "app_engagement_score" "preferred_channel"    "account_status"

[1] "driver_id"        "base_zone"        "employment_type"  "years_experience"
[5] "training_score"   "driver_rating"    "shift_preference" "active_flag"

[1] "vehicle_id"         "vehicle_type"       "assigned_zone"     
[4] "commission_date"    "battery_health_pct" "odometer_km"       
[7] "maintenance_status" "telematics_version"

[1] "incident_id"       "delivery_id"       "incident_type"    
[4] "reported_at"       "severity"          "resolution_status"
[7] "resolved_hours"

[1] "event_id"        "customer_id"     "order_id"        "event_timestamp"
 [5] "event_type"      "session_id"      "device_type"     "zone_context"   
 [9] "api_latency_ms"  "success_flag"

[1] "hub_id"         "hub_name"       "zone"           "hub_type"      
[5] "capacity_score"

This step checks the available columns in each dataset so that SQL queries can be written using the correct field names.

4. Preview the main datasets

In [ ]:
# Show first six rows of key datasets
head(orders)
head(deliveries)
head(complaints)
head(hubs)

,order_id,customer_id,service_type,order_created_at,promised_window_hours,pickup_zone,dropoff_zone,priority_level,order_value,booking_channel,special_handling_flag
,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<dbl>,<chr>,<int>
1,O00001,C0292,Passenger,2024-08-20 14:43:00,6,Airport,South,Medium,126.65,App,0
2,O00002,C0459,Passenger,2024-05-14 22:16:00,24,North,AIRPORT,Low,109.30,App,0
3,O00003,C0161,Passenger,2025-09-02 14:37:00,4,West,AIRPORT,High,33.50,Phone,0
4,O00004,C0520,Parcel,2025-01-11 17:15:00,2,RiverSide,North,Medium,10.04,App,1
5,O00005,C0558,Retail,2025-02-17 19:32:00,12,Riverside,SOUTH,Low,125.58,Phone,0
6,O00006,C0437,Retail,2024-08-05 04:55:00,1,CENTRAL,East,High,151.44,Web,1


,delivery_id,order_id,driver_id,vehicle_id,hub_id,dispatch_time,delivery_completed_at,delivery_status,route_distance_km,manual_route_override_count,proof_of_completion_missing,customer_rating_post_delivery,fuel_or_charge_cost
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<int>,<int>,<dbl>,<dbl>
1,DL00001,O00938,D004,V056,H05,2024-06-18 10:57:00,2024-06-19 09:05:59.904311,Failed,17.26,1,0,3.07,12.05
2,DL00002,O00004,D138,V007,H02,2025-01-11 18:45:00,2025-01-11 17:39:00.000000,OnTime,10.34,1,0,5.00,13.41
3,DL00003,O00639,D006,V049,H02,2025-06-02 20:39:00,2025-06-02 21:45:32.366770,OnTime,7.92,0,0,4.98,8.51
4,DL00004,O00313,D116,V055,H02,2024-03-08 23:31:00,2024-03-09 23:30:08.103702,Delayed,16.42,0,0,4.18,13.62
5,DL00005,O00844,D108,V034,H01,2025-09-21 11:43:00,2025-09-21 15:45:34.131056,OnTime,14.52,1,0,4.18,9.22
6,DL00006,O00029,D037,V098,H03,2024-09-11 12:40:00,2024-09-12 17:11:52.384869,Delayed,13.84,0,0,1.57,9.58


,complaint_id,customer_id,order_id,complaint_type,channel,severity,created_at,status,resolution_days,compensation_amount
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<dbl>
1,CP0001,C0464,O00814,AppIssue,App,High,2025-03-30 02:36:00,Open,11,23.99
2,CP0002,C0056,O00628,MissedPickup,Phone,Medium,2024-11-07 10:05:00,Open,4,21.64
3,CP0003,C0469,O00384,Delay,Chatbot,High,2024-01-02 15:47:00,Open,16,26.41
4,CP0004,C0631,O00406,Delay,App,Medium,2025-01-14 13:07:00,AwaitingCustomer,7,23.44
5,CP0005,C0535,O00154,Delay,Email,Medium,2024-08-31 05:56:00,Resolved,1,16.18
6,CP0006,C0096,O00147,Delay,App,Medium,2024-07-22 07:43:00,Resolved,9,18.51


,hub_id,hub_name,zone,hub_type,capacity_score
,<chr>,<chr>,<chr>,<chr>,<int>
1,H01,North Exchange,North,Dispatch,82
2,H02,South Link,South,Dispatch,78
3,H03,East Dock,East,Warehouse,74
4,H04,West Gate,West,Dispatch,69
5,H05,Central Core,Central,Control,88
6,H06,Airport Hub,Airport,Dispatch,71


This gives an initial preview of the data and helps confirm that the CSV files loaded correctly.

3. Total orders by service type

In [ ]:
# Query 1: Count number of orders for each service type
q1 <- sqldf("
  SELECT service_type,
         COUNT(*) AS total_orders
  FROM orders
  GROUP BY service_type
  ORDER BY total_orders DESC
")

# Display result
q1

service_type,total_orders
<chr>,<int>
Passenger,341
Parcel,308
Retail,297
Business,165
Medical,139


his query shows which service types generate the highest demand for NorthStar.

4. Orders by pickup and drop-off zone

In [ ]:
# Query 2: Count orders by pickup and drop-off zones
q2 <- sqldf("
  SELECT pickup_zone,
         dropoff_zone,
         COUNT(*) AS total_orders
  FROM orders
  GROUP BY pickup_zone, dropoff_zone
  ORDER BY total_orders DESC
")

# Display result
q2

pickup_zone,dropoff_zone,total_orders
<chr>,<chr>,<int>
East,South,17
South,West,12
WEST,WEST,11
CENTRAL,RiverSide,10
Central,WEST,10
EAST,Riverside,10
SOUTH,Riverside,10
South,AIRPORT,10
Airport,Airport,9


This query identifies the busiest movement patterns between zones, which helps understand where operational pressure is highest.

7. Average and total order value by service type

In [ ]:
# Query 3: Calculate average and total order value for each service type
q3 <- sqldf("
  SELECT service_type,
         ROUND(AVG(order_value), 2) AS avg_order_value,
         ROUND(SUM(order_value), 2) AS total_order_value
  FROM orders
  GROUP BY service_type
  ORDER BY total_order_value DESC
")

# Display result
q3

service_type,avg_order_value,total_order_value
<chr>,<dbl>,<dbl>
Passenger,96.07,32761.11
Parcel,87.62,26985.62
Retail,90.01,26734.06
Business,92.25,15220.43
Medical,87.14,12111.93


This query compares service types by revenue value, helping identify which services are financially more important.

8. Delivery status summary

In [ ]:
# Query 4: Count deliveries by delivery status
q4 <- sqldf("
  SELECT delivery_status,
         COUNT(*) AS total_deliveries
  FROM deliveries
  GROUP BY delivery_status
  ORDER BY total_deliveries DESC
")

# Display result
q4

delivery_status,total_deliveries
<chr>,<int>
OnTime,616
Delayed,202
Failed,132


This query gives a basic view of service outcomes, including completed, delayed, or failed deliveries.

JOINED SQL QUERIES

9. Orders joined with deliveries

In [ ]:
# Query 5: Join orders with deliveries to analyse service performance
q5 <- sqldf("
  SELECT o.service_type,
         d.delivery_status,
         COUNT(*) AS total_cases,
         ROUND(AVG(d.customer_rating_post_delivery), 2) AS avg_rating,
         ROUND(AVG(d.manual_route_override_count), 2) AS avg_route_overrides,
         ROUND(AVG(d.fuel_or_charge_cost), 2) AS avg_cost
  FROM orders o
  INNER JOIN deliveries d
    ON o.order_id = d.order_id
  GROUP BY o.service_type, d.delivery_status
  ORDER BY total_cases DESC
")

# Display result
q5

service_type,delivery_status,total_cases,avg_rating,avg_route_overrides,avg_cost
<chr>,<chr>,<int>,<dbl>,<dbl>,<dbl>
Passenger,OnTime,171,4.28,0.78,12.05
Parcel,OnTime,156,4.28,0.99,13.18
Retail,OnTime,146,4.28,0.98,12.84
Business,OnTime,73,4.31,1.10,12.78
Medical,OnTime,70,4.27,0.80,12.65
Passenger,Delayed,53,3.06,1.11,13.28
Retail,Delayed,50,3.14,0.82,12.72
Parcel,Delayed,49,3.10,1.37,12.94
Passenger,Failed,38,3.01,0.95,12.74


This query connects customer orders with delivery outcomes to show how each service type performs operationally.

10.Complaints joined with orders

In [ ]:
# Query 6: Join complaints with orders to analyse dissatisfaction by service type
q6 <- sqldf("
  SELECT o.service_type,
         c.complaint_type,
         c.severity,
         COUNT(*) AS complaint_count,
         ROUND(AVG(c.resolution_days), 2) AS avg_resolution_days,
         ROUND(AVG(c.compensation_amount), 2) AS avg_compensation
  FROM orders o
  INNER JOIN complaints c
    ON o.order_id = c.order_id
  GROUP BY o.service_type, c.complaint_type, c.severity
  ORDER BY complaint_count DESC
")

# Display result
q6

service_type,complaint_type,severity,complaint_count,avg_resolution_days,avg_compensation
<chr>,<chr>,<chr>,<int>,<dbl>,<dbl>
Retail,Delay,Medium,20,7.00,20.33
Passenger,Delay,Medium,14,7.36,15.81
Retail,MissedPickup,Medium,14,6.00,18.17
Parcel,DriverBehaviour,Medium,12,5.50,15.93
Business,Delay,Medium,10,3.70,19.43
Passenger,AppIssue,Medium,9,4.44,11.73
Parcel,Delay,Medium,8,4.50,19.64
Retail,AppIssue,Medium,8,6.75,15.22
Retail,Delay,Low,8,6.13,5.13


This query links complaints to service types, helping identify which services create the most customer dissatisfaction.

11. Deliveries joined with hubs

In [ ]:
# Query 7: Join deliveries with hubs to analyse hub and zone performance
q7 <- sqldf("
  SELECT h.zone,
         h.hub_name,
         d.delivery_status,
         COUNT(*) AS total_deliveries,
         ROUND(AVG(d.route_distance_km), 2) AS avg_distance,
         ROUND(AVG(d.fuel_or_charge_cost), 2) AS avg_cost,
         ROUND(AVG(d.customer_rating_post_delivery), 2) AS avg_rating
  FROM deliveries d
  INNER JOIN hubs h
    ON d.hub_id = h.hub_id
  GROUP BY h.zone, h.hub_name, d.delivery_status
  ORDER BY total_deliveries DESC
")

# Display result
q7

zone,hub_name,delivery_status,total_deliveries,avg_distance,avg_cost,avg_rating
<chr>,<chr>,<chr>,<int>,<dbl>,<dbl>,<dbl>
North,North Exchange,OnTime,93,13.99,12.86,4.26
East,East Dock,OnTime,85,14.03,12.59,4.21
West,West Gate,OnTime,83,12.58,12.79,4.32
Central,Midtown Relay,OnTime,80,12.13,11.40,4.32
Riverside,Riverside Hub,OnTime,76,14.09,12.79,4.25
South,South Link,OnTime,70,14.63,12.45,4.30
Central,Central Core,OnTime,67,15.12,13.59,4.19
Airport,Airport Hub,OnTime,62,14.03,13.14,4.43
West,West Gate,Delayed,28,16.13,14.66,3.21


This query compares hub and zone performance, which is important because the case study mentions that some locations perform worse than others.

12. Deliveries joined with incidents

In [ ]:
# Query 8: Join deliveries with incidents to find hidden operational problems
q8 <- sqldf("
  SELECT d.delivery_status,
         COUNT(i.incident_id) AS total_incidents,
         COUNT(DISTINCT d.delivery_id) AS total_deliveries
  FROM deliveries d
  LEFT JOIN incidents i
    ON d.delivery_id = i.delivery_id
  GROUP BY d.delivery_status
  ORDER BY total_incidents DESC
")

# Display result
q8

delivery_status,total_incidents,total_deliveries
<chr>,<int>,<int>
OnTime,191,616
Delayed,55,202
Failed,34,132


This query checks whether delivery outcomes are linked with incidents, showing hidden failures that may not be visible from delivery status alone.

13. Customer complaints and customer type

In [ ]:
colnames(customers)

[1] "customer_id"          "age"                  "home_zone"           
[4] "customer_type"        "signup_date"          "loyalty_score"       
[7] "app_engagement_score" "preferred_channel"    "account_status"

In [ ]:
# Query 9: Join customers with complaints using correct column name
q9 <- sqldf("
  SELECT cu.customer_type,
         c.complaint_type,
         c.severity,
         COUNT(*) AS complaint_count
  FROM customers cu
  INNER JOIN complaints c
    ON cu.customer_id = c.customer_id
  GROUP BY cu.customer_type, c.complaint_type, c.severity
  ORDER BY complaint_count DESC
")

# Display result
q9

customer_type,complaint_type,severity,complaint_count
<chr>,<chr>,<chr>,<int>
Consumer,Delay,Medium,45
Consumer,MissedPickup,Medium,28
Consumer,DriverBehaviour,Medium,22
Consumer,Delay,Low,20
Consumer,AppIssue,Medium,18
Consumer,DriverBehaviour,High,14
Consumer,MissedPickup,High,14
Consumer,AppIssue,Low,11
Consumer,Delay,High,11


The query was updated after checking the dataset structure. The correct column name is 'customer_type' instead of 'customer_segment'. This highlights the importance of validating real dataset fields before writing SQL queries.

14. Driver performance and delivery outcomes

In [ ]:
colnames(drivers)

[1] "driver_id"        "base_zone"        "employment_type"  "years_experience"
[5] "training_score"   "driver_rating"    "shift_preference" "active_flag"

In [ ]:
# Query 10: Join drivers with deliveries using correct column
q10 <- sqldf("
  SELECT dr.employment_type,
         d.delivery_status,
         COUNT(*) AS total_deliveries,
         ROUND(AVG(d.customer_rating_post_delivery), 2) AS avg_rating,
         ROUND(AVG(d.manual_route_override_count), 2) AS avg_route_overrides
  FROM drivers dr
  INNER JOIN deliveries d
    ON dr.driver_id = d.driver_id
  GROUP BY dr.employment_type, d.delivery_status
  ORDER BY total_deliveries DESC
")

q10

employment_type,delivery_status,total_deliveries,avg_rating,avg_route_overrides
<chr>,<chr>,<int>,<dbl>,<dbl>
FullTime,OnTime,376,4.29,0.97
PartTime,OnTime,158,4.31,0.76
FullTime,Delayed,121,3.11,1.08
FullTime,Failed,85,3.06,1.16
Contract,OnTime,82,4.22,0.99
PartTime,Delayed,55,3.15,1.07
PartTime,Failed,29,2.95,1.00
Contract,Delayed,26,3.06,1.04
Contract,Failed,18,3.17,0.50


The query was corrected after examining the drivers dataset. The correct column name is 'employment_type' instead of 'contract_type'. This step ensures accurate SQL execution based on actual dataset structure.